# SingBERT Inference

This notebook only loads a saved classifier from the `model/` folder and tests it on a few sample messages.


In [2]:
# %pip install -q transformers torch

from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_DIR = Path("model")
MAX_LENGTH = 128

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Could not find model folder: {MODEL_DIR.resolve()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

id2label = model.config.id2label
if not id2label:
    id2label = {i: f"LABEL_{i}" for i in range(model.config.num_labels)}
else:
    id2label = {int(k): v for k, v in id2label.items()}

print(f"Loaded model from: {MODEL_DIR.resolve()}")
print(f"Device: {device}")
print(f"Labels: {id2label}")


/Users/jithinbathula/Documents/Academic/Final Term/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1297.07it/s]

Loaded model from: /Users/jithinbathula/Documents/Academic/Final Term/AI/model
Device: cpu
Labels: {0: 'authority_scam', 1: 'job_scam', 2: 'love_scam', 3: 'not_scam'}


In [3]:
def classify(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probabilities = torch.softmax(logits, dim=-1).squeeze(0).cpu()
    predicted_id = int(torch.argmax(probabilities).item())

    return {
        "text": text,
        "prediction": id2label[predicted_id],
        "probabilities": {id2label[i]: round(float(probabilities[i].item()), 4) for i in range(len(id2label))},
    }


In [4]:
sample_messages = [
    "Eh bro, want to go makan later at the hawker centre?",
    "Congratulations! You have been selected for a $5000 reward. Click here to claim now.",
    "Officer here. Your bank account is under investigation and you must respond immediately.",
    "Hey darling, I miss you so much. Can you send me money for flight ticket to come see you?",
]

for message in sample_messages:
    result = classify(message)
    print(f"Text:       {result['text']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Probs:      {result['probabilities']}")
    print()


Text:       Eh bro, want to go makan later at the hawker centre?
Prediction: not_scam
Probs:      {'authority_scam': 0.0013, 'job_scam': 0.0011, 'love_scam': 0.0012, 'not_scam': 0.9964}

Text:       Congratulations! You have been selected for a $5000 reward. Click here to claim now.
Prediction: authority_scam
Probs:      {'authority_scam': 0.8038, 'job_scam': 0.1779, 'love_scam': 0.0102, 'not_scam': 0.0081}

Text:       Officer here. Your bank account is under investigation and you must respond immediately.
Prediction: authority_scam
Probs:      {'authority_scam': 0.9941, 'job_scam': 0.0021, 'love_scam': 0.0024, 'not_scam': 0.0014}

Text:       Hey darling, I miss you so much. Can you send me money for flight ticket to come see you?
Prediction: love_scam
Probs:      {'authority_scam': 0.0021, 'job_scam': 0.0044, 'love_scam': 0.9892, 'not_scam': 0.0043}

